In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# define metadata
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
batch_size = 64

transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_data = datasets.CIFAR100(root="./data", download=True, train=True, transform=transforms)
test_data = datasets.CIFAR100(root="./data", download=True, train=False, transform=transforms)

train_dataloader = DataLoader(dataset=train_data, batch_size=batch_size, num_workers=2, shuffle=True)
test_dataloader = DataLoader(dataset=test_data, batch_size=batch_size, num_workers=2, shuffle=False)

def count_parameters(model, trainable=True):
    if trainable:
        params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    else:
        params = sum(p.numel() for p in model.parameters())
    return params

Using device: cuda


100%|██████████| 169M/169M [01:07<00:00, 2.51MB/s] 


In [2]:
def train_model(model , epochs= 10, train_dataloader = train_dataloader, test_dataloader = test_dataloader):
    loss_hist = []
    val_acc_hist = []
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        in_epoch_loss = 0.0
        for x, y in train_dataloader:
            x, y = x.to(device), y.to(device)
            y_hat = model(x)

            loss = criterion(y_hat, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            in_epoch_loss += loss.item()

        train_loss = in_epoch_loss / len(train_dataloader)

        total = 0
        correct = 0

        model.eval()
        with torch.no_grad():
            for x, y in test_dataloader:
                x, y = x.to(device), y.to(device)
                y_hat = model(x)
                _, predicted = torch.max(y_hat.data, 1)
                total += y.size(0)
                correct += (predicted == y).sum().item()

        accuracy = 100 * correct / total

        loss_hist.append(train_loss)
        val_acc_hist.append(accuracy)
        print(f"[Epoch {epoch + 1}/{epochs}] Training Loss {train_loss:.4f} | Test Set Validation Acc: {accuracy:.4f}")

    return loss_hist, val_acc_hist

In [4]:
import torchvision.models as models

num_classes = 100

resnet18 = models.resnet18(weights=None)
resnet18.fc = nn.Linear(resnet18.fc.in_features, num_classes)
resnet18.to(device)

print(f"resnet18 size: {count_parameters(resnet18)}")

resnet_loss_hist, resnet_val_acc_hist = train_model(resnet18, epochs=80)

resnet18 size: 11227812
[Epoch 1/80] Training Loss 3.5318 | Test Set Validation Acc: 24.0400
[Epoch 2/80] Training Loss 2.7804 | Test Set Validation Acc: 31.0100
[Epoch 3/80] Training Loss 2.3809 | Test Set Validation Acc: 37.1300
[Epoch 4/80] Training Loss 2.0947 | Test Set Validation Acc: 41.2400
[Epoch 5/80] Training Loss 1.8314 | Test Set Validation Acc: 43.1100
[Epoch 6/80] Training Loss 1.5829 | Test Set Validation Acc: 43.9200
[Epoch 7/80] Training Loss 1.3357 | Test Set Validation Acc: 46.1500
[Epoch 8/80] Training Loss 1.0798 | Test Set Validation Acc: 45.2100
[Epoch 9/80] Training Loss 0.8416 | Test Set Validation Acc: 46.1300
[Epoch 10/80] Training Loss 0.6408 | Test Set Validation Acc: 44.9800
[Epoch 11/80] Training Loss 0.4882 | Test Set Validation Acc: 45.0000
[Epoch 12/80] Training Loss 0.3865 | Test Set Validation Acc: 44.6000
[Epoch 13/80] Training Loss 0.3278 | Test Set Validation Acc: 44.3500
[Epoch 14/80] Training Loss 0.2855 | Test Set Validation Acc: 43.1000
[Epoc

In [9]:
efficientnet = models.efficientnet_b3()

print(f"efficient net b0 size: {count_parameters(efficientnet)}")

efficient net b0 size: 12233232
